# Function 5: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [1]:
import numpy as np

input_data = np.load('../../data/initial_data/function_5/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.278167, 0.217734, 0.996929, 0.992772],
    [0.331025, 0.592662, 0.991036, 0.994402],
    [0.5, 0.5, 0.5, 0.5]
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


Before: (20, 4)
After: (23, 4)
[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.278167   0.217734   0.996

In [ ]:
output_data = np.load('../../data/initial_data/function_5/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    1552.6699801013826,
    1803.9236591562037,
    -0.015979341188442648
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


Before: (20,)
After: (22,)
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+00
 2.58370525e+02 7.84343889e+01 5.75715369e+01 1.09571876e+02
 8.84799176e+00 2.33223610e+02 2.44230883e+01 6.44201468e+01
 6.34767158e+01 7.97291299e+01 3.55806818e+02 1.08885962e+03
 2.88667516e+01 4.51815703e+01 4.31612757e+02 9.97233189e+00
 1.55266998e+03 1.80392366e+03]


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5, 0.5]])
actual_output = 32.0025

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 5
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


Function 5, dimension d=4
Data shape: (23, 4) (23,)
Current best observed y: 0.1129397953712203
Current best x: [0.43834987 0.8043397  0.21024527 0.15129482]


In [4]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


,x1,x2,x3,x4,y,log_abs_y,rank_min
2,0.438350,0.804340,0.210245,0.151295,0.112940,-2.180900,1
3,0.706051,0.534192,0.264243,0.482088,4.210898,1.437676,2
8,0.153786,0.729382,0.422598,0.443074,8.847992,2.180191,3
19,0.126885,0.153430,0.770162,0.190518,9.972332,2.299814,4
1,0.758653,0.536518,0.656000,0.360342,18.301380,2.906976,5
10,0.677491,0.358510,0.479592,0.072880,24.423088,3.195529,6
16,0.725262,0.479870,0.088947,0.759760,28.866752,3.362690,7
22,0.500000,0.500000,0.500000,0.500000,32.002500,3.465814,8
17,0.355482,0.639619,0.417618,0.122604,45.181570,3.810689,9
6,0.553621,0.667350,0.323806,0.814870,57.571537,4.053028,10


## Neural-network surrogate + input gradients

In [5]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Fit a small neural network to y. If the output scale is extreme, use log|y| instead.
# This cell automatically chooses log|y| if the ratio of magnitudes is very large or if values are near zero.
X = input_data.astype(np.float32)
y_raw = output_data.astype(np.float32)
use_log_abs = (np.nanmax(np.abs(y_raw)) / max(np.nanmin(np.abs(y_raw) + 1e-300), 1e-300) > 1e4)

y_target = np.log(np.abs(y_raw) + 1e-300).astype(np.float32) if use_log_abs else y_raw.copy()
target_name = "log_abs_y" if use_log_abs else "y"

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X).astype(np.float32)
y_scaled = y_scaler.fit_transform(y_target.reshape(-1, 1)).astype(np.float32).ravel()

X_t = torch.tensor(X_scaled, dtype=torch.float32)
y_t = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32)

class SurrogateNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        width = max(16, 4*d)
        self.net = nn.Sequential(
            nn.Linear(d, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, 1)
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
model = SurrogateNN(d)
opt = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(3000):
    opt.zero_grad()
    pred = model(X_t)
    loss = loss_fn(pred, y_t)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_scaled = model(X_t).numpy().ravel()
    pred_target = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

print("Target modelled:", target_name)
print(f"In-sample MSE in target space: {mean_squared_error(y_target, pred_target):.6g}")
print(f"In-sample R² in target space: {r2_score(y_target, pred_target):.4f}")


Target modelled: log_abs_y
In-sample MSE in target space: 2.09484e-09
In-sample R² in target space: 1.0000


In [6]:
# Compute gradients of the network prediction with respect to original input variables.
X_grad = torch.tensor(X_scaled, dtype=torch.float32, requires_grad=True)
pred = model(X_grad)

# Sum is used so autograd gives one gradient per input point.
pred.sum().backward()
grad_scaled = X_grad.grad.detach().numpy()

# Convert from scaled-input / scaled-output gradient to original input scale.
grad_target = grad_scaled * (y_scaler.scale_[0] / x_scaler.scale_)
grad_norm = np.linalg.norm(grad_target, axis=1)

grad_df = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
grad_df["y"] = output_data
grad_df[target_name] = y_target
for j in range(d):
    grad_df[f"grad_x{j+1}"] = grad_target[:, j]
grad_df["grad_norm"] = grad_norm
grad_df["dominant_variable"] = [f"x{np.argmax(np.abs(row))+1}" for row in grad_target]

display(grad_df.sort_values("grad_norm", ascending=False))

avg_abs_grad = np.mean(np.abs(grad_target), axis=0)
print("Average absolute gradient:")
for j, val in enumerate(avg_abs_grad):
    print(f"x{j+1}: {val:.6g}")
print("Most influential variable on average:", f"x{np.argmax(avg_abs_grad)+1}")


,x1,x2,x3,x4,y,log_abs_y,grad_x1,grad_x2,grad_x3,grad_x4,grad_norm,dominant_variable
6,0.553621,0.667350,0.323806,0.814870,57.571537,4.053028,-6.324536,-12.706430,5.571337,19.511798,24.762939,x4
14,0.438933,0.774092,0.378167,0.933696,355.806818,5.874388,-6.494085,-11.970075,4.499372,14.055859,20.081518,x4
3,0.706051,0.534192,0.264243,0.482088,4.210898,1.437676,-3.445000,-18.081212,4.724315,5.023716,19.655918,x2
21,0.331025,0.592662,0.991036,0.994402,1803.923659,7.497719,-1.534731,0.077461,-5.397957,14.570683,15.614229,x4
17,0.355482,0.639619,0.417618,0.122604,45.181570,3.810689,-2.176286,-12.535827,8.588633,-2.424955,15.541178,x2
8,0.153786,0.729382,0.422598,0.443074,8.847992,2.180191,-0.830237,-12.389896,9.043494,1.808924,15.467896,x2
9,0.463442,0.630025,0.107906,0.957644,233.223610,5.451998,-3.541200,-4.401343,7.685669,10.281436,14.024598,x4
13,0.511142,0.817957,0.728710,0.112354,79.729130,4.378635,-1.748200,-6.693403,10.718078,2.482413,12.996054,x3
16,0.725262,0.479870,0.088947,0.759760,28.866752,3.362690,-1.893330,-8.970362,5.121530,7.491052,12.899537,x2
2,0.438350,0.804340,0.210245,0.151295,0.112940,-2.180900,-2.554471,-8.957172,7.467725,-1.134294,11.992072,x2


Average absolute gradient:
x1: 2.3347
x2: 5.89805
x3: 4.0765
x4: 5.79912
Most influential variable on average: x2


In [7]:
# Use the neural network to propose a point by searching random candidates.
# For minimisation, choose low predicted target. If target is log_abs_y, this finds small magnitude, not necessarily negative y.
rng = np.random.default_rng(2)
candidates = rng.random((30000 if d <= 4 else 60000, d)).astype(np.float32)
cand_scaled = x_scaler.transform(candidates).astype(np.float32)
with torch.no_grad():
    pred_scaled = model(torch.tensor(cand_scaled)).numpy().ravel()
    pred_target = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

nn_results = pd.DataFrame(candidates, columns=[f"x{i+1}" for i in range(d)])
nn_results[f"pred_{target_name}"] = pred_target
nn_results = nn_results.sort_values(f"pred_{target_name}", ascending=True)
display(nn_results.head(10))

best_nn = nn_results.iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(float)
print("NN suggested point:", np.round(best_nn, 6))
print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(best_nn)))


,x1,x2,x3,x4,pred_log_abs_y
24689,0.887869,0.771635,0.000332,0.284945,-4.003727
1082,0.878715,0.784989,0.006187,0.331916,-3.983881
26259,0.984940,0.742004,0.007497,0.270537,-3.861580
22848,0.797842,0.807310,0.006860,0.230362,-3.846028
9450,0.998244,0.776051,0.015675,0.379925,-3.797373
16553,0.856980,0.823359,0.038460,0.257809,-3.783818
6100,0.957991,0.740194,0.032278,0.282869,-3.775797
28333,0.724890,0.803352,0.054670,0.322944,-3.761958
27731,0.951117,0.766624,0.032805,0.399847,-3.724251
15747,0.791113,0.804004,0.071785,0.374389,-3.723156


NN suggested point: [8.87869e-01 7.71635e-01 3.32000e-04 2.84945e-01]
Portal format: x1=0.887869, x2=0.771635, x3=0.000332, x4=0.284945
